In [0]:
# Imports

from pyspark.sql.functions import col, trim, to_date, expr, when, current_timestamp
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number
from pyspark.sql.functions import col

In [0]:
# Ler a Bronze V2
df_bronze = spark.table("workspace.drs_bronze.votacao_secao_2024_go")

In [0]:
# Padronizando os nomes das colunas
df_eleicoes = df_bronze \
    .withColumnRenamed("dt_geracao", "generation_date") \
    .withColumnRenamed("ano_eleicao", "election_year") \
    .withColumnRenamed("cd_tipo_eleicao", "election_type_code") \
    .withColumnRenamed("nr_turno", "round_number") \
    .withColumnRenamed("cd_eleicao", "election_code") \
    .withColumnRenamed("ds_eleicao", "election_description") \
    .withColumnRenamed("dt_eleicao", "election_date") \
    .withColumnRenamed("tp_abrangencia", "scope_type") \
    .withColumnRenamed("sg_uf", "state_abbreviation") \
    .withColumnRenamed("sg_ue", "electoral_unit_abbreviation") \
    .withColumnRenamed("cd_municipio", "city_code") \
    .withColumnRenamed("nm_municipio", "city_name") \
    .withColumnRenamed("nr_zona", "zone_number") \
    .withColumnRenamed("nr_secao", "section_number") \
    .withColumnRenamed("cd_cargo", "office_code") \
    .withColumnRenamed("ds_cargo", "office_description") \
    .withColumnRenamed("nr_votavel", "ballot_number") \
    .withColumnRenamed("nm_votavel", "ballot_item_name") \
    .withColumnRenamed("qt_votos", "vote_count") \
    .withColumnRenamed("nr_local_votacao", "polling_place_number") \
    .withColumnRenamed("sq_candidato", "candidate_sequence_id") \
    .withColumnRenamed("nm_local_votacao", "polling_place_name")


print(f"Total registros: {df_eleicoes.count()}")


In [0]:
# Transformar/Convert colunas para o padarão para depois salvar a Silver V2

from pyspark.sql.functions import col, trim, to_date, expr, when

df_silver = (
    df_eleicoes
    .withColumn("election_date", to_date(col("election_date"), "M/d/yyyy"))
    )

In [0]:
# criando o campo source_file
from pyspark.sql.functions import lit

df = df_silver.withColumn("source_file", lit("votacao_secao_2024_GO.csv"))

In [0]:
# Separando registros inválidos para quarentena

df_invalid = df.filter("""
sg_uf != "go"
OR nr_zona IS NULL
OR qt_votos IS NULL
""")

print(f"Total registros inválidos: {df_invalid.count()}")

In [0]:
# Salvando quarentena da Silver V2

df_invalid.write \
    .format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable("workspace.drs_silver.votacao_secao_2024_go_quarantine")


In [0]:
# criando o campo ingestion_timestamp
from pyspark.sql.functions import current_timestamp

df_silver = df.withColumn("ingestion_timestamp", current_timestamp())

In [0]:
# Filtrando valores válidos
from pyspark.sql.functions import col

df_silver_filtered = (
    df_silver.dropDuplicates()
      .filter(
          (col("generation_date").isNotNull()) &
          (col("vote_count").isNotNull()) &
          (col("section_number").isNotNull()) 
      )
)

In [0]:
# Criando o campo silver_processed_timestamp em df_silver_v2
from pyspark.sql.functions import current_timestamp

df_eleicoes = df_silver_filtered \
.withColumn("silver_processed_timestamp", current_timestamp()) \
.withColumn("created_at", current_timestamp()) \
.withColumn("updated_at", current_timestamp())

In [0]:
# Grantindo a deduplicação por chave de negócio

window_spec = Window.partitionBy(
    "generation_date",
    "section_number",
    "office_code",
).orderBy(
    col("ingestion_timestamp").desc()
)

df_eleicoes = (
    df_eleicoes
    .withColumn("row_num", row_number().over(window_spec))
    .filter(col("row_num") == 1)
    .drop("row_num")
)

In [0]:
# Validando shema da dataframe v2
df_eleicoes.printSchema()

In [0]:
# Salvando a tabela Silver em uma staging table

df_eleicoes.write \
.format("delta") \
.mode("overwrite") \
.option("overwriteSchema", "true") \
.saveAsTable("workspace.drs_silver.votacao_secao_2024_go_staging")

In [0]:

# Craindo tabela final se não existir

spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.drs_silver.votacao_secao_2024_go
AS
SELECT * FROM workspace.drs_silver.votacao_secao_2024_go_staging WHERE 1 = 0
""")

In [0]:
# Validar schema da staging

spark.table("workspace.drs_silver.votacao_secao_2024_go_staging").printSchema()

In [0]:
# Salvando tabela drs_silver_v2 com MERGE

spark.sql("""
MERGE INTO workspace.drs_silver.votacao_secao_2024_go AS target
USING workspace.drs_silver.votacao_secao_2024_go_staging AS source

ON target.city_code = source.city_code
AND target.zone_number = source.zone_number
AND target.section_number = source.section_number
AND target.office_code = source.office_code
AND target.ballot_number = source.ballot_number

WHEN MATCHED THEN
UPDATE SET
    target.generation_date = source.generation_date,
    target.hh_geracao = source.hh_geracao,
    target.election_year = source.election_year,
    target.election_type_code = source.election_type_code,
    target.nm_tipo_eleicao = source.nm_tipo_eleicao,
    target.round_number = source.round_number,
    target.election_code = source.election_code,
    target.election_description = source.election_description,
    target.election_date = source.election_date,
    target.scope_type = source.scope_type,
    target.state_abbreviation = source.state_abbreviation,
    target.electoral_unit_abbreviation = source.electoral_unit_abbreviation,
    target.nm_ue = source.nm_ue,
    target.city_name = source.city_name,
    target.office_description = source.office_description,
    target.ballot_item_name = source.ballot_item_name,
    target.vote_count = source.vote_count,
    target.polling_place_number = source.polling_place_number,
    target.candidate_sequence_id = source.candidate_sequence_id,
    target.polling_place_name = source.polling_place_name,
    target.ds_local_votacao_endereco = source.ds_local_votacao_endereco,
    target.ingestion_timestamp = source.ingestion_timestamp,
    target.source_file = source.source_file,
    target.silver_processed_timestamp = source.silver_processed_timestamp,
    target.updated_at = current_timestamp()

WHEN NOT MATCHED THEN
INSERT (
    generation_date,
    hh_geracao,
    election_year,
    election_type_code,
    nm_tipo_eleicao,
    round_number,
    election_code,
    election_description,
    election_date,
    scope_type,
    state_abbreviation,
    electoral_unit_abbreviation,
    nm_ue,
    city_code,
    city_name,
    zone_number,
    section_number,
    office_code,
    office_description,
    ballot_number,
    ballot_item_name,
    vote_count,
    polling_place_number,
    candidate_sequence_id,
    polling_place_name,
    ds_local_votacao_endereco,
    ingestion_timestamp,
    source_file,
    silver_processed_timestamp,
    created_at,
    updated_at
)
VALUES (
    source.generation_date,
    source.hh_geracao,
    source.election_year,
    source.election_type_code,
    source.nm_tipo_eleicao,
    source.round_number,
    source.election_code,
    source.election_description,
    source.election_date,
    source.scope_type,
    source.state_abbreviation,
    source.electoral_unit_abbreviation,
    source.nm_ue,
    source.city_code,
    source.city_name,
    source.zone_number,
    source.section_number,
    source.office_code,
    source.office_description,
    source.ballot_number,
    source.ballot_item_name,
    source.vote_count,
    source.polling_place_number,
    source.candidate_sequence_id,
    source.polling_place_name,
    source.ds_local_votacao_endereco,
    source.ingestion_timestamp,
    source.source_file,
    source.silver_processed_timestamp,
    current_timestamp(),
    current_timestamp()
)
""")

In [0]:
# Validando a tabela drs_silver_votacao_secao_2024_go

spark.sql("""
SELECT
* 
--COUNT(*) AS total_rows
FROM workspace.drs_silver.votacao_secao_2024_go
""")


In [0]:
# Verificar duplicidade pela chave do merge

spark.sql("""
SELECT
    generation_date,
    section_number,
    office_code,
    COUNT(*) AS qtd
FROM workspace.drs_silver.votacao_secao_2024_go
GROUP BY generation_date, section_number, office_code 
HAVING COUNT(*) > 1
ORDER BY qtd DESC
""").display()

In [0]:
# Data Quality checks da Silver V2
dq = spark.sql("""
SELECT
    SUM(CASE WHEN vote_count < 0 THEN 1 ELSE 0 END) AS vote_count,
    SUM(CASE WHEN generation_date IS NULL THEN 1 ELSE 0 END) AS generation_date,
    SUM(CASE WHEN election_date IS NULL THEN 1 ELSE 0 END) AS election_date,
    SUM(CASE WHEN ballot_number is NULL THEN 1 ELSE 0 END) AS ballot_number
FROM workspace.drs_silver.votacao_secao_2024_go
""")

display(dq)

In [0]:
# validação quarentena
spark.sql("""
SELECT COUNT(*) AS total_invalid
FROM workspace.drs_silver.votacao_secao_2024_go_quarantine
""").display()

In [0]:
# Falhar notebook se houver erro crítico de qualidade
dq_row = dq.collect()[0]

if (
    dq_row["vote_count"] > 0 or
    dq_row["ballot_number"] > 0 
):
    raise Exception("Data Quality check failed in workspace.drs_silver.votacao_secao_2024_go")
